In [ ]:
import os
import random
import numpy as np
import torch
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

from functions.running import one_run, plot_results, plot_running_time


def seed_everything(seed: int = 0) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True


def run(X, y, dataname: str) -> None:
    for init_type in ['he', 'xavier', 'orthogonal']:
        epochs = 200
        batch_size = 64
        n_layer = 5
        dataname_suffix = f"{dataname}{n_layer}{init_type}"
        nruns = 10
        results = []
        timing = []
        output_img_dir = os.path.join('output', 'img')
        output_res_dir = os.path.join('output', 'res')
        output_time_dir = os.path.join('output', 'time')
        os.makedirs(output_img_dir, exist_ok=True)
        os.makedirs(output_res_dir, exist_ok=True)
        os.makedirs(output_time_dir, exist_ok=True)
        for i in range(nruns):
            seed_everything(seed=i)
            run_res_path = os.path.join(output_res_dir, f"{dataname_suffix}_run{i}.npy")
            run_time_path = os.path.join(output_time_dir, f"{dataname_suffix}_time_run{i}.npy")
            if os.path.exists(run_res_path) and os.path.exists(run_time_path):
                res_array = np.load(run_res_path, allow_pickle=True)
                time_array = np.load(run_time_path, allow_pickle=True)
                if res_array.dtype == object:
                    res_array = np.stack([np.array(r) for r in res_array])
                if time_array.dtype == object:
                    time_array = np.stack([np.array(t) for t in time_array])
            else:
                res_one_run, time_one_run = one_run(init_type, X, y, epochs, batch_size, n_layer,
                                                   n_frozen_epochs=200, dataname=dataname)
                res_array = np.stack(res_one_run)
                time_array = np.stack(time_one_run)
                np.save(run_res_path, res_array)
                np.save(run_time_path, time_array)
            results.append(res_array)
            timing.append(time_array)
        results = np.stack(results)
        timing = np.stack(timing)
        mean_all_run = np.mean(results, axis=0)
        std_all_run = np.std(results, axis=0, ddof=1)
        ci_all_run = 1.96 * std_all_run / np.sqrt(nruns)
        mean_time_all_run = np.mean(timing, axis=0)
        plot_results(dataname_suffix, mean_all_run, ci_all_run, output_dir=output_img_dir)
        np.save(os.path.join(output_res_dir, dataname_suffix), results)
        np.save(os.path.join(output_time_dir, dataname_suffix), timing)
        plot_running_time(f"{dataname_suffix}_time", mean_time_all_run, output_dir=output_img_dir)

In [4]:
use_gpu = torch.cuda.is_available()
if use_gpu:
    print(f"GPU is available. Using {torch.cuda.get_device_name(0)}")
else:
    print("GPU was requested but is not available. Using CPU instead.")


GPU is available. Using Tesla T4


In [ ]:
data = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.train', header = None,sep=',')

test = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.test',
                     header=None, sep = ',')
data = pd.concat([data, test])
data = data.to_numpy()
X,y = data[:,1:], data[:,0]
G = len(np.unique(y))
X = X.astype('float')
run(X, y, 'heart')

In [ ]:
data = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.train', header = None,sep=',')
print(data.head())
test = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.test',
                     header=None, sep = ',')
data = pd.concat([data, test])
data = data.to_numpy()
X,y = data[:,1:], data[:,0]
G = len(np.unique(y))
print(np.shape(X))
for g in range(G):
  print(sum(y==g))
X.shape
X = X.astype('float')

run(X, y, 'micromass')

In [ ]:
from sklearn.preprocessing import LabelEncoder
data = pd.read_csv('http://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data',
                  sep = ",", header = None)
data = pd.DataFrame.to_numpy(data)
X, y = data[:,:34].astype(np.float64), data[:,34]
le2 = LabelEncoder()
y = le2.fit_transform(y)
G = len(np.unique(y))
X = np.delete(X,[0,1], axis = 1)
for g in range(G):
  print(sum(y==g))
X.shape

run(X, y, 'ionosphere')

In [ ]:
data = pd.read_csv('/kaggle/input/running-dataset/pd_speech_features.csv', sep = ",", header = [0,1])
print(data.shape)
data.head()
data = data.to_numpy()
X, y = data[:,:-1], data[:,-1]
G = len(np.unique(y))
print('input shape:', X.shape)

run(X, y, 'parkinson')

In [ ]:
features = pd.read_csv('/kaggle/input/running-dataset/features.csv')

# Separate features and labels
X = features.iloc[:, 2:54]
y = features.iloc[:, 1]

# Convert string labels to integer indices
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Pass the encoded labels to your experiment
run(X, y_encoded, 'htad')

In [ ]:
run(X=None, y=None, 'mnist')

In [ ]:
run(X=None, y=None, 'cifar10')